# Live League Games

In [1]:
%load_ext autoreload
%autoreload 2
import requests
import pandas as pd
from datetime import datetime as dt
from sqlalchemy.engine import create_engine, URL
from utils import unix_to_datetime 
import numpy as np
import json
from retry import retry 
pd.set_option("display.max_columns", 100)
pd.set_option('display.max_rows', 50)

In [2]:
# Steam API constants
STEAM_URL = 'http://api.steampowered.com/'
LIVE_LEAGUE_GAMES = 'IDOTA2Match_570/GetLiveLeagueGames/v1'
REAL_TIME_STATS = 'IDOTA2MatchStats_570/GetRealtimeStats/v1' # Requires server_steam_id 
API_KEY = 'F0E5D7D11B592792FE20D84FBB745D97'

### Fetching premium and professional leagues' match details

In [3]:
# Function to retrieve Json from Steam WebAPI
session = requests.Session()
session.params.update({'key': API_KEY})

@retry(tries=3, delay=2)
def fetch_live_league_games():
    try:
        url = f'{STEAM_URL}{LIVE_LEAGUE_GAMES}'
        res = session.get(url)
        match_details = res.json()
        if not match_details:
            raise ValueError("Empty dictionary, retrying...")
        else:
            return match_details
    except Exception as err:
        print("Did not get a response, retrying...")
        raise

In [4]:
# output file suffix
current_datetime = dt.now().strftime('%Y%m%d')
current_datetime 

'20231002'

In [5]:
game_data = fetch_live_league_games()
games = game_data['result']['games']

In [6]:
games_df = pd.DataFrame(games)
games_df

,players,dire_team,lobby_id,match_id,spectators,league_id,league_node_id,stream_delay_s,radiant_series_wins,dire_series_wins,series_type,scoreboard,radiant_team
0,"[{'account_id': 292950831, 'name': 'FACEIT.com...","{'team_name': 'Äijie kans endaa', 'team_id': 9...",28457202964673246,7362498983,4,4122,0,120,0,0,0,"{'duration': 2494.933349609375, 'roshan_respaw...",NaN
1,"[{'account_id': 292954476, 'name': 'FACEIT.com...","{'team_name': 'Kelan Rotat', 'team_id': 883825...",28457202982902251,7362529093,1,4122,0,120,0,0,0,"{'duration': 945.6666870117188, 'roshan_respaw...","{'team_name': 'Äijie kans endaa', 'team_id': 9..."
2,"[{'account_id': 292954709, 'name': 'FACEIT.com...","{'team_name': 'Team Another Alone', 'team_id':...",28457202976180671,7362516040,1,4122,0,120,0,0,0,"{'duration': 1906.0335693359375, 'roshan_respa...","{'team_name': 'TeamSilencs', 'team_id': 525851..."
3,"[{'account_id': 292949272, 'name': 'FACEIT.com...","{'team_name': 'KaWoRi Fan-Club', 'team_id': 91...",28457203001635132,7362558124,1,4122,0,120,0,0,0,"{'duration': 0, 'roshan_respawn_timer': 0, 'ra...","{'team_name': '40 Min. Midas Casino', 'team_id..."
4,"[{'account_id': 292952156, 'name': 'FACEIT.com...","{'team_name': 'TCL United', 'team_id': 9182232...",28457202979702590,7362523850,1,4122,0,120,0,0,0,"{'duration': 1133.2000732421875, 'roshan_respa...",NaN
5,"[{'account_id': 292956839, 'name': 'FACEIT.com...","{'team_name': 'Safas o te Avanzo', 'team_id': ...",28457202993936341,7362543729,1,4122,0,120,0,0,0,"{'duration': 749, 'roshan_respawn_timer': 0, '...",NaN
6,"[{'account_id': 292959003, 'name': 'FACEIT.com...",NaN,28457202984352436,7362531712,0,4122,0,120,0,0,0,"{'duration': 769.4666748046875, 'roshan_respaw...",NaN
7,"[{'account_id': 292952842, 'name': 'FACEIT.com...","{'team_name': 'Повітрювач Освітря ', 'team_id'...",28457203002257628,7362558834,0,4122,0,120,0,0,0,"{'duration': 0, 'roshan_respawn_timer': 0, 'ra...",NaN
8,"[{'account_id': 292960436, 'name': 'FACEIT.com...",NaN,28457202989860634,7362537549,0,4122,0,120,0,0,0,"{'duration': 557.3999633789062, 'roshan_respaw...",NaN
9,"[{'account_id': 344153979, 'name': 'Lucky_crYa...","{'team_name': 'CrimsonSky', 'team_id': 9025669...",28457202916174994,7362541546,2,15660,51,120,0,0,1,"{'duration': 455.73333740234375, 'roshan_respa...","{'team_name': 'Moonwalkers', 'team_id': 897006..."


In [7]:
# Import the list of premium and professional league games id

import yaml

file_path = 'league_ids.yml'

with open(file_path, 'r') as file:
    content = yaml.safe_load(file) or {}
    if 'PREMIUM_LEAGUES' in content:
        premium_leagues = content['PREMIUM_LEAGUES']
    if 'PROFESSIONAL_LEAGUES' in content:
        professional_leagues = content['PROFESSIONAL_LEAGUES']
        
premium_list = list(premium_leagues.values())
professional_list = list(professional_leagues.values())



In [8]:
def live_match_template():
    template = {
        'match_id': np.nan,
        'radiant_team_id': np.nan,
        'radiant_name': np.nan,
        'dire_team_id': np.nan,
        'dire_name': np.nan,
        'game_duration': 0,
        'start_time': np.nan,
        'radiant_win' : -1
    }
    
    for i in list(range(0, 5)) + list(range(128, 133)):
        template[f"{i}_account_id"] = np.nan
        template[f"{i}_hero_id"] = np.nan
        
    return template


In [9]:
live_league_games = []

for row in games:
    
    league_id = row.get('league_id', np.nan)
    if league_id in premium_list + professional_list:
        
        match_dict = live_match_template()
        
        # Populate common fields
        match_dict['league_id'] = league_id
        match_dict['match_id'] = row.get('match_id', np.nan)
        match_dict['radiant_team_id'] = row.get('radiant_team', {}).get('team_id', np.nan)
        match_dict['radiant_name'] = row.get('radiant_team', {}).get('team_name', np.nan)
        match_dict['dire_team_id'] = row.get('dire_team', {}).get('team_id', np.nan)
        match_dict['dire_name'] = row.get('dire_team', {}).get('team_name', np.nan)
        match_dict['game_duration'] = row.get('scoreboard', {}).get('duration', np.nan)
        match_dict['start_time'] = dt.now().timestamp()
        
        # Populate player data
        for team in ['radiant', 'dire']:
            for player in row.get('scoreboard', {}).get(team, {}).get('players', []):
                slot = player.get('player_slot', None)
                if slot is not None:  # To ensure only valid slots are updated
                    match_dict[f"{slot}_account_id"] = player.get('account_id', np.nan)
                    match_dict[f"{slot}_hero_id"] = player.get('hero_id', np.nan)
                    live_league_games.append(match_dict)

    
if len(live_league_games) == 0:
    print("No premium or professional games right now")
else:
    print(len(live_league_games))
    print(live_league_games)   


10
[{'match_id': 7362515574, 'radiant_team_id': 9104067, 'radiant_name': 'Rakuzan', 'dire_team_id': 8533458, 'dire_name': 'Team Flamingos', 'game_duration': 1584.933349609375, 'start_time': 1696278135.477426, 'radiant_win': -1, '0_account_id': 1479535614, '0_hero_id': 119, '1_account_id': 1550368099, '1_hero_id': 83, '2_account_id': 1266748819, '2_hero_id': 23, '3_account_id': 169786155, '3_hero_id': 138, '4_account_id': 131774502, '4_hero_id': 67, '128_account_id': 2491317, '128_hero_id': 47, '129_account_id': 339941742, '129_hero_id': 135, '130_account_id': 377568540, '130_hero_id': 37, '131_account_id': 187885221, '131_hero_id': 17, '132_account_id': 1247302157, '132_hero_id': 137, 'league_id': 14915}, {'match_id': 7362515574, 'radiant_team_id': 9104067, 'radiant_name': 'Rakuzan', 'dire_team_id': 8533458, 'dire_name': 'Team Flamingos', 'game_duration': 1584.933349609375, 'start_time': 1696278135.477426, 'radiant_win': -1, '0_account_id': 1479535614, '0_hero_id': 119, '1_account_id':

In [10]:
live_league_games

[{'match_id': 7362515574,
  'radiant_team_id': 9104067,
  'radiant_name': 'Rakuzan',
  'dire_team_id': 8533458,
  'dire_name': 'Team Flamingos',
  'game_duration': 1584.933349609375,
  'start_time': 1696278135.477426,
  'radiant_win': -1,
  '0_account_id': 1479535614,
  '0_hero_id': 119,
  '1_account_id': 1550368099,
  '1_hero_id': 83,
  '2_account_id': 1266748819,
  '2_hero_id': 23,
  '3_account_id': 169786155,
  '3_hero_id': 138,
  '4_account_id': 131774502,
  '4_hero_id': 67,
  '128_account_id': 2491317,
  '128_hero_id': 47,
  '129_account_id': 339941742,
  '129_hero_id': 135,
  '130_account_id': 377568540,
  '130_hero_id': 37,
  '131_account_id': 187885221,
  '131_hero_id': 17,
  '132_account_id': 1247302157,
  '132_hero_id': 137,
  'league_id': 14915},
 {'match_id': 7362515574,
  'radiant_team_id': 9104067,
  'radiant_name': 'Rakuzan',
  'dire_team_id': 8533458,
  'dire_name': 'Team Flamingos',
  'game_duration': 1584.933349609375,
  'start_time': 1696278135.477426,
  'radiant_win

In [11]:
json_string = json.dumps(live_league_games)
with open(f'live_league_games_{current_datetime}.json','w') as file:
    file.write(json_string)

In [12]:
df = pd.read_json(json_string, convert_dates=False)

In [13]:
df

,match_id,radiant_team_id,radiant_name,dire_team_id,dire_name,game_duration,start_time,radiant_win,0_account_id,0_hero_id,1_account_id,1_hero_id,2_account_id,2_hero_id,3_account_id,3_hero_id,4_account_id,4_hero_id,128_account_id,128_hero_id,129_account_id,129_hero_id,130_account_id,130_hero_id,131_account_id,131_hero_id,132_account_id,132_hero_id,league_id
0,7362515574,9104067,Rakuzan,8533458,Team Flamingos,1584.93335,1.696278e+09,-1,1479535614,119,1550368099,83,1266748819,23,169786155,138,131774502,67,2491317,47,339941742,135,377568540,37,187885221,17,1247302157,137,14915
1,7362515574,9104067,Rakuzan,8533458,Team Flamingos,1584.93335,1.696278e+09,-1,1479535614,119,1550368099,83,1266748819,23,169786155,138,131774502,67,2491317,47,339941742,135,377568540,37,187885221,17,1247302157,137,14915
2,7362515574,9104067,Rakuzan,8533458,Team Flamingos,1584.93335,1.696278e+09,-1,1479535614,119,1550368099,83,1266748819,23,169786155,138,131774502,67,2491317,47,339941742,135,377568540,37,187885221,17,1247302157,137,14915
3,7362515574,9104067,Rakuzan,8533458,Team Flamingos,1584.93335,1.696278e+09,-1,1479535614,119,1550368099,83,1266748819,23,169786155,138,131774502,67,2491317,47,339941742,135,377568540,37,187885221,17,1247302157,137,14915
4,7362515574,9104067,Rakuzan,8533458,Team Flamingos,1584.93335,1.696278e+09,-1,1479535614,119,1550368099,83,1266748819,23,169786155,138,131774502,67,2491317,47,339941742,135,377568540,37,187885221,17,1247302157,137,14915
5,7362515574,9104067,Rakuzan,8533458,Team Flamingos,1584.93335,1.696278e+09,-1,1479535614,119,1550368099,83,1266748819,23,169786155,138,131774502,67,2491317,47,339941742,135,377568540,37,187885221,17,1247302157,137,14915
6,7362515574,9104067,Rakuzan,8533458,Team Flamingos,1584.93335,1.696278e+09,-1,1479535614,119,1550368099,83,1266748819,23,169786155,138,131774502,67,2491317,47,339941742,135,377568540,37,187885221,17,1247302157,137,14915
7,7362515574,9104067,Rakuzan,8533458,Team Flamingos,1584.93335,1.696278e+09,-1,1479535614,119,1550368099,83,1266748819,23,169786155,138,131774502,67,2491317,47,339941742,135,377568540,37,187885221,17,1247302157,137,14915
8,7362515574,9104067,Rakuzan,8533458,Team Flamingos,1584.93335,1.696278e+09,-1,1479535614,119,1550368099,83,1266748819,23,169786155,138,131774502,67,2491317,47,339941742,135,377568540,37,187885221,17,1247302157,137,14915
9,7362515574,9104067,Rakuzan,8533458,Team Flamingos,1584.93335,1.696278e+09,-1,1479535614,119,1550368099,83,1266748819,23,169786155,138,131774502,67,2491317,47,339941742,135,377568540,37,187885221,17,1247302157,137,14915


In [14]:
from preprocessing import preprocess_df

In [15]:
live_df = preprocess_df(df)
live_df

,0_hero_id,1_hero_id,2_hero_id,3_hero_id,4_hero_id,128_hero_id,129_hero_id,130_hero_id,131_hero_id,132_hero_id,0_account_id,1_account_id,2_account_id,3_account_id,4_account_id,128_account_id,129_account_id,130_account_id,131_account_id,132_account_id,radiant_name,dire_name,radiant_win,start_time,match_id
0,119,83,23,138,67,47,135,37,17,137,1479535614,1550368099,1266748819,169786155,131774502,2491317,339941742,377568540,187885221,1247302157,Rakuzan,Team Flamingos,-1,2023-10-02 13:22:15.477426,7362515574


In [16]:
num_live_df = len(live_df) # variable to be uses as limit_to arguments for all feature engineering scripts
num_live_df

1

### Get the historical df

In [17]:
url_object = URL.create(
    "postgresql+psycopg2",
    username='liuhaochen',
    host='localhost',
    port='5432',
    database='test'
)

engine = create_engine(url_object)

In [18]:
df = pd.read_sql(
    "SELECT * FROM pro_matches",
    con=engine
)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58902 entries, 0 to 58901
Data columns (total 28 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   0_hero_id        58766 non-null  float64
 1   1_hero_id        58769 non-null  float64
 2   2_hero_id        58773 non-null  float64
 3   3_hero_id        58769 non-null  float64
 4   4_hero_id        58769 non-null  float64
 5   128_hero_id      58766 non-null  float64
 6   129_hero_id      58771 non-null  float64
 7   130_hero_id      58780 non-null  float64
 8   131_hero_id      58771 non-null  float64
 9   132_hero_id      58775 non-null  float64
 10  0_account_id     58766 non-null  float64
 11  1_account_id     58769 non-null  float64
 12  2_account_id     58773 non-null  float64
 13  3_account_id     58769 non-null  float64
 14  4_account_id     58769 non-null  float64
 15  128_account_id   58766 non-null  float64
 16  129_account_id   58771 non-null  float64
 17  130_account_

In [19]:
df_historical = preprocess_df(df)

In [20]:
df_combined = pd.concat([live_df, df_historical])
df_combined

/var/folders/y7/llh2lh591fd8649s5dlf79_c0000gn/T/ipykernel_94918/2957358276.py:1: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  df_combined = pd.concat([live_df, df_historical])


,0_hero_id,1_hero_id,2_hero_id,3_hero_id,4_hero_id,128_hero_id,129_hero_id,130_hero_id,131_hero_id,132_hero_id,0_account_id,1_account_id,2_account_id,3_account_id,4_account_id,128_account_id,129_account_id,130_account_id,131_account_id,132_account_id,radiant_name,dire_name,radiant_win,start_time,match_id
0,119.0,83.0,23.0,138.0,67.0,47.0,135.0,37.0,17.0,137.0,1.479536e+09,1.550368e+09,1.266749e+09,1.697862e+08,1.317745e+08,2491317.0,339941742.0,3.775685e+08,1.878852e+08,1.247302e+09,Rakuzan,Team Flamingos,-1,2023-10-02 13:22:15.477426,7.362516e+09
0,83.0,120.0,138.0,101.0,38.0,79.0,121.0,29.0,109.0,106.0,1.174215e+08,2.941354e+08,8.790180e+08,1.812673e+08,2.285167e+08,105045291.0,104334048.0,1.388806e+08,1.772040e+08,2.624760e+08,StoRm,Luna Galaxy,True,2023-09-30 09:52:54.000000,7.359096e+09
1,119.0,80.0,13.0,107.0,5.0,111.0,86.0,54.0,28.0,135.0,9.170671e+08,1.251984e+09,1.282465e+09,8.806954e+08,1.482586e+09,339941742.0,2491317.0,1.087461e+09,1.247302e+09,1.878852e+08,Ghost Sheep,Team Flamingos,True,2023-09-30 09:37:19.000000,7.359076e+09
2,80.0,101.0,20.0,106.0,114.0,99.0,107.0,56.0,59.0,5.0,1.247302e+09,2.491317e+06,3.399417e+08,1.087461e+09,1.878852e+08,917067055.0,880695375.0,1.251984e+09,1.282465e+09,1.482586e+09,Team Flamingos,Ghost Sheep,False,2023-09-30 08:55:16.000000,7.359003e+09
3,51.0,44.0,43.0,23.0,101.0,83.0,98.0,10.0,121.0,29.0,1.050453e+08,1.772040e+08,1.388806e+08,2.624760e+08,1.043340e+08,117421467.0,294135421.0,8.790180e+08,1.812673e+08,2.285167e+08,Luna Galaxy,StoRm,False,2023-09-30 08:51:13.000000,7.358997e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56480,58.0,98.0,19.0,42.0,111.0,41.0,68.0,96.0,128.0,46.0,1.264698e+08,1.182073e+08,2.305808e+08,3.185882e+08,7.660036e+07,874542740.0,67893845.0,9.315945e+07,7.981801e+07,1.727400e+08,Wind and Rain,felt,False,2021-05-17 17:02:06.000000,5.999283e+09
56481,126.0,78.0,109.0,128.0,102.0,86.0,96.0,106.0,94.0,68.0,8.745427e+08,9.315945e+07,1.727400e+08,7.981801e+07,6.789384e+07,126469785.0,118207269.0,2.305808e+08,3.185882e+08,7.660036e+07,felt,Wind and Rain,False,2021-05-17 16:02:25.000000,5.999250e+09
56482,96.0,123.0,15.0,121.0,72.0,23.0,2.0,94.0,128.0,30.0,1.182073e+08,1.264698e+08,2.305808e+08,7.660036e+07,3.185882e+08,874542740.0,93159453.0,1.727400e+08,7.981801e+07,6.789384e+07,Wind and Rain,felt,False,2021-05-17 15:02:01.000000,5.999214e+09
56483,68.0,58.0,97.0,106.0,19.0,111.0,17.0,61.0,88.0,6.0,8.681866e+07,9.163191e+08,1.261746e+08,8.861133e+07,2.038514e+08,97366926.0,97072681.0,7.126641e+07,1.841317e+08,1.075799e+08,Infinity,Crewmates,True,2021-05-17 14:40:35.000000,5.999202e+09


### Feature Engineering

In [21]:
from team_features import create_team_level_features
from player_hero_features import create_player_hero_features
from heroes_features import create_hero_level_features

In [22]:

df_combined

,0_hero_id,1_hero_id,2_hero_id,3_hero_id,4_hero_id,128_hero_id,129_hero_id,130_hero_id,131_hero_id,132_hero_id,0_account_id,1_account_id,2_account_id,3_account_id,4_account_id,128_account_id,129_account_id,130_account_id,131_account_id,132_account_id,radiant_name,dire_name,radiant_win,start_time,match_id
0,119.0,83.0,23.0,138.0,67.0,47.0,135.0,37.0,17.0,137.0,1.479536e+09,1.550368e+09,1.266749e+09,1.697862e+08,1.317745e+08,2491317.0,339941742.0,3.775685e+08,1.878852e+08,1.247302e+09,Rakuzan,Team Flamingos,-1,2023-10-02 13:22:15.477426,7.362516e+09
0,83.0,120.0,138.0,101.0,38.0,79.0,121.0,29.0,109.0,106.0,1.174215e+08,2.941354e+08,8.790180e+08,1.812673e+08,2.285167e+08,105045291.0,104334048.0,1.388806e+08,1.772040e+08,2.624760e+08,StoRm,Luna Galaxy,True,2023-09-30 09:52:54.000000,7.359096e+09
1,119.0,80.0,13.0,107.0,5.0,111.0,86.0,54.0,28.0,135.0,9.170671e+08,1.251984e+09,1.282465e+09,8.806954e+08,1.482586e+09,339941742.0,2491317.0,1.087461e+09,1.247302e+09,1.878852e+08,Ghost Sheep,Team Flamingos,True,2023-09-30 09:37:19.000000,7.359076e+09
2,80.0,101.0,20.0,106.0,114.0,99.0,107.0,56.0,59.0,5.0,1.247302e+09,2.491317e+06,3.399417e+08,1.087461e+09,1.878852e+08,917067055.0,880695375.0,1.251984e+09,1.282465e+09,1.482586e+09,Team Flamingos,Ghost Sheep,False,2023-09-30 08:55:16.000000,7.359003e+09
3,51.0,44.0,43.0,23.0,101.0,83.0,98.0,10.0,121.0,29.0,1.050453e+08,1.772040e+08,1.388806e+08,2.624760e+08,1.043340e+08,117421467.0,294135421.0,8.790180e+08,1.812673e+08,2.285167e+08,Luna Galaxy,StoRm,False,2023-09-30 08:51:13.000000,7.358997e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56480,58.0,98.0,19.0,42.0,111.0,41.0,68.0,96.0,128.0,46.0,1.264698e+08,1.182073e+08,2.305808e+08,3.185882e+08,7.660036e+07,874542740.0,67893845.0,9.315945e+07,7.981801e+07,1.727400e+08,Wind and Rain,felt,False,2021-05-17 17:02:06.000000,5.999283e+09
56481,126.0,78.0,109.0,128.0,102.0,86.0,96.0,106.0,94.0,68.0,8.745427e+08,9.315945e+07,1.727400e+08,7.981801e+07,6.789384e+07,126469785.0,118207269.0,2.305808e+08,3.185882e+08,7.660036e+07,felt,Wind and Rain,False,2021-05-17 16:02:25.000000,5.999250e+09
56482,96.0,123.0,15.0,121.0,72.0,23.0,2.0,94.0,128.0,30.0,1.182073e+08,1.264698e+08,2.305808e+08,7.660036e+07,3.185882e+08,874542740.0,93159453.0,1.727400e+08,7.981801e+07,6.789384e+07,Wind and Rain,felt,False,2021-05-17 15:02:01.000000,5.999214e+09
56483,68.0,58.0,97.0,106.0,19.0,111.0,17.0,61.0,88.0,6.0,8.681866e+07,9.163191e+08,1.261746e+08,8.861133e+07,2.038514e+08,97366926.0,97072681.0,7.126641e+07,1.841317e+08,1.075799e+08,Infinity,Crewmates,True,2021-05-17 14:40:35.000000,5.999202e+09


In [26]:
team_level_features = create_team_level_features(df_combined, num_live_df)
heroes_features = create_hero_level_features(df_combined, num_live_df)
player_hero_feature = create_player_hero_features(df_combined, num_live_df)

/Users/liuhaochen/Projects/projects/DotaMatchPredictor/player_hero_features.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_subset['win_rate'] = df_subset.apply(lambda row: last_10_matches_winrate(df_combined, row[TIME_COL], row['account_id'], row['hero_id']), axis=1)
/Users/liuhaochen/Projects/projects/DotaMatchPredictor/player_hero_features.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_subset['player_hero_win_rate_col'] = df_subset['player_num'].astype(str) + '_account_ ' + df_subset['he

In [27]:
df_final = pd.merge(pd.merge(team_level_features,heroes_features,how='inner', on=['match_id','radiant_win','start_time']),player_hero_feature, how='inner', on='match_id')
df_final

,radiant_dire_matchup,radiant_win_rate,dire_win_rate,radiant_win,start_time,match_id,Abaddon,Alchemist,Ancient Apparition,Anti-Mage,Arc Warden,Axe,Bane,Batrider,Beastmaster,Bloodseeker,Bounty Hunter,Brewmaster,Bristleback,Broodmother,Centaur Warrunner,Chaos Knight,Chen,Clinkz,Clockwerk,Crystal Maiden,Dark Seer,Dark Willow,Dawnbreaker,Dazzle,Death Prophet,Disruptor,Doom,Dragon Knight,Drow Ranger,Earth Spirit,Earthshaker,Elder Titan,Ember Spirit,Enchantress,Enigma,Faceless Void,Grimstroke,Gyrocopter,Hoodwink,Huskar,Invoker,Io,Jakiro,Juggernaut,...,Rubick,Sand King,Shadow Demon,Shadow Fiend,Shadow Shaman,Silencer,Skywrath Mage,Slardar,Slark,Snapfire,Sniper,Spectre,Spirit Breaker,Storm Spirit,Sven,Techies,Templar Assassin,Terrorblade,Tidehunter,Timbersaw,Tinker,Tiny,Treant Protector,Troll Warlord,Tusk,Underlord,Undying,Ursa,Vengeful Spirit,Venomancer,Viper,Visage,Void Spirit,Warlock,Weaver,Windranger,Winter Wyvern,Witch Doctor,Wraith King,Zeus,0_account_ 0_hero_win_rate,128_account_ 128_hero_win_rate,129_account_ 129_hero_win_rate,130_account_ 130_hero_win_rate,131_account_ 131_hero_win_rate,132_account_ 132_hero_win_rate,1_account_ 1_hero_win_rate,2_account_ 2_hero_win_rate,3_account_ 3_hero_win_rate,4_account_ 4_hero_win_rate
0,0.7,0.8,0.9,-1,2023-10-02 13:22:15.477426,7.362516e+09,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0.7,0.5,0.7,0.777778,0.0,0.8,0.9,0.714286,0.7,0.5


In [29]:
df_final.to_csv(f"df_live_{current_datetime}.csv", index=False)